In [4]:
import pandas as pd
from pathlib import Path

In [5]:
DATA_DIR = Path('/Users/ben/Projects/ADEPT/Data')

In [13]:
eval_df = pd.read_excel(DATA_DIR / 'ADEPTselected 25records-validation.xlsx', sheet_name='ADEPTselected spp')

In [27]:
eval_df["taxon"] = eval_df["taxon"].astype("string").str.lower()

In [42]:
eval_df

,taxon,life form,habitat,habit,clonality,perennial organ,plant min. height [m],plant max. height [m],indumentum,spinescence,...,seed min. length [mm],seed max. length [mm],seed min. diameter [mm],seed max. diameter [mm],dispersal mode,ploidy level (2n),root min. depth [cm],root max. depth [cm],source,source_id
0,achillea millefolium,perennial,NaN,erect leafy,"rhizomatous, stoloniferous","stem, stolon, rhizome",0.06,0.65,"woolly, tomentose, glabrous",NaN,...,NaN,NaN,NaN,NaN,anemochory,"2n = 18, 27, 36, 45, 54, 63, 72",NaN,NaN,efloras.flora_of_north_america,NaN
1,achillea millefolium,perennial,NaN,"erect leafy/tussock, herb, tree/shrub, erect l...","solitary plant, rhizome","stem, rhizome, trunk",0.40 m,1.00 m,"villous, villouse, glandular",NaN,...,2 mm,2 mm,NaN,NaN,NaN,"2n = 18, 27, 36",NaN,NaN,efloras.flora_of_china,NaN
2,achillea millefolium,NaN,NaN,shrub,solitary plant,trunk,1 m,1 m,"pubescent, woolly, hairs absent, hairy",NaN,...,2.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,efloras.flora_of_pakistan,NaN
3,achillea millefolium,perennial,NaN,"herb, scrambler, erect leafy",NaN,"stem, tap root",0.5 m,0.5 m,pubescentpinulose,spinulatepiny,...,1.5 mm,2.0 mm,NaN,NaN,NaN,NaN,NaN,NaN,bhl,2652962.0
4,alchemilla alpina,NaN,NaN,"tree/shrub, erect leafy",solitary plant,"trunk, stem",0.05 m,0.20 m,"pubescent, sericeous, hairy, glabrous",NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,efloras.flora_of_north_america,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
60,vaccinium myrtillus,NaN,NaN,NaN,rhizomatous,rhizome,NaN,NaN,glandular,NaN,...,NaN,NaN,1.0,1.0,NaN,"2n = 24, 48",NaN,NaN,efloras.flora_of_north_america,NaN
61,vaccinium myrtillus,perennial,NaN,shrub,"solitary plant, rhizomatous",rhizome,0.15 m,0.60 m,NaN,NaN,...,NaN,NaN,NaN,6.0,10,2n = 24,NaN,NaN,efloras.flora_of_china,NaN
62,vaccinium myrtillus,NaN,NaN,shrub,solitary plant,trunk,1.5 m [0.15],3 m [0.40],"glandular, hairy, ciliate",NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,bhl,30218234.0
63,valeriana dioica,NaN,NaN,erect leafy,"solitary plant, rhizome","rhizome, tap root",0.11 m,0.38 m,NaN,NaN,...,2.5 mm,3 mm,NaN,NaN,NaN,NaN,NaN,NaN,efloras.flora_of_pakistan,NaN


In [23]:
eval_df.source_id.unique()

array([      nan,  2652962., 47353191.,   750379.,  1115231., 41685059.,
       22883499.,  7934755., 30046404., 11320960.,  8010221., 11836210.,
       29987781.,  8884077.,  7934510., 47370362.,  4614077., 31968879.,
       30156396., 46694846., 61885166., 58253841., 30218234., 33362730.])

In [39]:
pred_xslx = DATA_DIR / 'uk-peatland-species-traits.xlsx'

pred_df_list = []

for source, group_df in eval_df.groupby("source"):
    df = pd.read_excel(pred_xslx, sheet_name=source)

    df["taxon"] = df["taxon"].astype("string").str.lower()

    

    if source == 'bhl':

        matched_df = df.merge(
            group_df[["taxon", "source_id"]].drop_duplicates(),
            on=["taxon", "source_id"],
            how="inner"
        )

    else:

        matched_df = df[df.taxon.isin(group_df.taxon)]

    pred_df_list.append(matched_df)


In [40]:
pred_df = pd.concat(pred_df_list, ignore_index=True)

In [48]:
def get_row_key(row):
    parts = [row.taxon, row.source]
    if source_id := row.get("source_id", None):
        parts.append(str(source_id))
    
    return '|'.join(parts)


eval_df['key'] = eval_df.apply(get_row_key,  axis=1)    

In [49]:
pred_df['key'] = pred_df.apply(get_row_key,  axis=1)  

In [68]:
pred_df[pred_df['perennial organ'].notna()]

,taxon,life form,habitat,habit,clonality,perennial organ,plant min. height [m],plant max. height [m],indumentum,spinescence,...,seed min. length [mm],seed max. length [mm],seed min. diameter [mm],seed max. diameter [mm],dispersal mode,ploidy level (2n),root min. depth [cm],root max. depth [cm],source,source_id
key,,,,,,,,,,,,,,,,,,,,,


In [60]:
eval_df[eval_df.key.isin(pred_df['key'])]

,taxon,life form,habitat,habit,clonality,perennial organ,plant min. height [m],plant max. height [m],indumentum,spinescence,...,seed max. length [mm],seed min. diameter [mm],seed max. diameter [mm],dispersal mode,ploidy level (2n),root min. depth [cm],root max. depth [cm],source,source_id,key
0,achillea millefolium,perennial,NaN,erect leafy,"rhizomatous, stoloniferous","stem, stolon, rhizome",0.06,0.65,"woolly, tomentose, glabrous",NaN,...,NaN,NaN,NaN,anemochory,"2n = 18, 27, 36, 45, 54, 63, 72",NaN,NaN,efloras.flora_of_north_america,NaN,achillea millefolium|efloras.flora_of_north_am...
1,achillea millefolium,perennial,NaN,"erect leafy/tussock, herb, tree/shrub, erect l...","solitary plant, rhizome","stem, rhizome, trunk",0.40 m,1.00 m,"villous, villouse, glandular",NaN,...,2 mm,NaN,NaN,NaN,"2n = 18, 27, 36",NaN,NaN,efloras.flora_of_china,NaN,achillea millefolium|efloras.flora_of_china|nan
2,achillea millefolium,NaN,NaN,shrub,solitary plant,trunk,1 m,1 m,"pubescent, woolly, hairs absent, hairy",NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,efloras.flora_of_pakistan,NaN,achillea millefolium|efloras.flora_of_pakistan...
3,achillea millefolium,perennial,NaN,"herb, scrambler, erect leafy",NaN,"stem, tap root",0.5 m,0.5 m,pubescentpinulose,spinulatepiny,...,2.0 mm,NaN,NaN,NaN,NaN,NaN,NaN,bhl,2652962.0,achillea millefolium|bhl|2652962.0
4,alchemilla alpina,NaN,NaN,"tree/shrub, erect leafy",solitary plant,"trunk, stem",0.05 m,0.20 m,"pubescent, sericeous, hairy, glabrous",NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,efloras.flora_of_north_america,NaN,alchemilla alpina|efloras.flora_of_north_ameri...
6,alchemilla filicaulis,NaN,NaN,"erect leafy, tree/shrub",solitary plant,"stem, trunk",NaN,0.4,"pubescent, hairy",NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,efloras.flora_of_north_america,NaN,alchemilla filicaulis|efloras.flora_of_north_a...
8,andromeda polifolia,NaN,NaN,"erect leafy, scrambler",NaN,tap root,0.05 m,0.80 m,"pubescent, hairy",NaN,...,1 mm,NaN,NaN,NaN,2n = 48,NaN,NaN,efloras.flora_of_north_america,NaN,andromeda polifolia|efloras.flora_of_north_ame...
9,andromeda polifolia,NaN,NaN,shrub,solitary plant,trunk,1,3,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,bhl,1115231.0,andromeda polifolia|bhl|1115231.0
10,antennaria dioica,NaN,NaN,NaN,"stoloniferous, stolon, stolons",stolon,0.03,0.1,"papillose, glabrous, tomentose",NaN,...,NaN,NaN,NaN,NaN,2n = 28,NaN,NaN,efloras.flora_of_north_america,NaN,antennaria dioica|efloras.flora_of_north_ameri...
12,betula nana,NaN,NaN,"climber/scrambler, tree/shrub, shrub, scrambler",solitary plant,"stolon, trunk",NaN,1,pubescent,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,efloras.flora_of_north_america,NaN,betula nana|efloras.flora_of_north_america|nan


In [61]:
eval_df = eval_df.set_index('key')

In [64]:
eval_df.to_csv('eval.csv')

In [65]:
pred_df = pred_df.set_index('key')

In [66]:
pred_df.to_csv('pred.csv')

In [51]:
bhl_df = pd.read_excel(pred_xslx, sheet_name='bhl')

In [59]:
bhl_df[bhl_df.taxon == 'Salix reticulata']

,taxon,life form,habitat,habit,clonality,perennial organ,plant min. height [m],plant max. height [m],indumentum,spinescence,...,seed min. length [mm],seed max. length [mm],seed min. diameter [mm],seed max. diameter [mm],dispersal mode,ploidy level (2n),root min. depth [cm],root max. depth [cm],source,source_id
1607,Salix reticulata,NaN,NaN,shrub,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,bhl,804944
1608,Salix reticulata,NaN,NaN,shrub,NaN,NaN,NaN,NaN,"glabrous, pubescent, sericeous",NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,bhl,878190
1609,Salix reticulata,NaN,NaN,shrub,NaN,NaN,NaN,NaN,"glabrous, pubescent, sericeous",NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,bhl,4614417
1610,Salix reticulata,NaN,NaN,shrub,NaN,NaN,NaN,NaN,"glabrous, pubescent, sericeous",NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,bhl,18715378
1611,Salix reticulata,NaN,NaN,NaN,solitary plant,NaN,NaN,NaN,"tomentose, glabrous, pubescent",NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,bhl,22138604
1612,Salix reticulata,NaN,NaN,shrub,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,bhl,22883490
1613,Salix reticulata,NaN,NaN,NaN,solitary plant,NaN,NaN,NaN,"tomentose, glabrous, pubescent",NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,bhl,22887737
1614,Salix reticulata,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"tomentose, glabrous, pubescent",NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,bhl,23567788
1615,Salix reticulata,NaN,NaN,shrub,NaN,NaN,NaN,NaN,"woolly, pubescent, hairs absent",NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,bhl,26150943
1616,Salix reticulata,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"hairy, glabrous, sericeous",NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,bhl,30156057


In [82]:
combined = pd.read_excel(pred_xslx, sheet_name='combined')

In [76]:
combined[~combined['fruit type'].isna()]['fruit type']

29           nut
44    schizocarp
Name: fruit type, dtype: object

In [75]:


eval_df[~eval_df['fruit type'].isna()]['fruit type']

key
achillea millefolium|efloras.flora_of_north_america|nan           cypsela
achillea millefolium|efloras.flora_of_china|nan                    achene
achillea millefolium|efloras.flora_of_pakistan|nan                cypsela
achillea millefolium|bhl|2652962.0                                 achene
andromeda polifolia|efloras.flora_of_north_america|nan            capsule
antennaria dioica|efloras.flora_of_north_america|nan              cypsela
antennaria dioica|bhl|41685059.0                                   achene
betula nana|efloras.flora_of_north_america|nan                     samara
betula nana|bhl|22883499.0                                            nut
carex capillaris|efloras.flora_of_north_america|nan                achene
carex capillaris|efloras.flora_of_china|nan                      Nutlets 
carex capillaris|bhl|7934755.0                                     achene
drosera anglica|efloras.flora_of_north_america|nan                capsule
eleocharis palustris|efloras.flora

In [88]:
eval_df[eval_df.taxon == 'vaccinium myrtillus']

,taxon,life form,habitat,habit,clonality,perennial organ,plant min. height [m],plant max. height [m],indumentum,spinescence,...,seed min. length [mm],seed max. length [mm],seed min. diameter [mm],seed max. diameter [mm],dispersal mode,ploidy level (2n),root min. depth [cm],root max. depth [cm],source,source_id
key,,,,,,,,,,,,,,,,,,,,,
vaccinium myrtillus|efloras.flora_of_north_america|nan,vaccinium myrtillus,NaN,NaN,NaN,rhizomatous,rhizome,NaN,NaN,glandular,NaN,...,NaN,NaN,1.0,1.0,NaN,"2n = 24, 48",NaN,NaN,efloras.flora_of_north_america,NaN
vaccinium myrtillus|efloras.flora_of_china|nan,vaccinium myrtillus,perennial,NaN,shrub,"solitary plant, rhizomatous",rhizome,0.15 m,0.60 m,NaN,NaN,...,NaN,NaN,NaN,6.0,10,2n = 24,NaN,NaN,efloras.flora_of_china,NaN
vaccinium myrtillus|bhl|30218234.0,vaccinium myrtillus,NaN,NaN,shrub,solitary plant,trunk,1.5 m [0.15],3 m [0.40],"glandular, hairy, ciliate",NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,bhl,30218234.0


In [98]:
bhl_df = pd.read_excel(pred_xslx, sheet_name='bhl')

bhl_df[(bhl_df.taxon == 'Eleocharis palustris') & (bhl_df.source_id ==11320960)].values

array([['Eleocharis palustris', nan, nan, nan, nan, nan, nan, nan, nan,
        nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
        nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
        nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
        nan, nan, nan, 3.0, nan, nan, nan, nan, nan, nan, nan, nan, nan,
        nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
        nan, nan, nan, nan, nan, nan, nan, nan, 'bhl', 11320960]],
      dtype=object)